In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import json
from python.functions.bridge import parse_quarter, build_bridge_inputs, run_bridge

In [ ]:
B = build_bridge_inputs()

rdl  = run_bridge("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "ensemble_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"])
vecm = run_bridge("../../data/outputs/forecasts/vecm_unconditional_forecast.csv", "vecm_log",
                   B["seed"], B["p"], B["seasonal"], B["fy_map"], B["net_add"], B["actual_back"], strip_space=True)

target = 1_500_000
for name, d in [("ARDL/NARDL ensemble", rdl), ("VECM (unconditional)", vecm)]:
    print(name)
    for fy, v in d.items():
        print(f"{fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")


non-private new build is held flat at its recent average (it won't respond to the policy scenarios), and conversions/change-of-use/demolitions are likewise held at 2021-24 averages. 

In [ ]:
import matplotlib.pyplot as plt

fy_start = lambda s: int(str(s)[:4])  # "2022-23" -> 2022

# Actual history from LT120, plus the 2024-25 actual already in delivery
actual = B["lt120"]["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)
if 2024 not in actual.index:
    actual.loc[2024] = rdl["2024-25"]
actual = actual.sort_index()

# Forecast path (2025-26 onward)
def fcast_series(d):
    s = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) >= 2025}).sort_index()
    return pd.concat([actual.iloc[[-1]], s])   # prepend last actual to close the gap

# Prepend the last actual point so the forecast line connects without a gap
join_rdl  = fcast_series(rdl)
join_vecm = fcast_series(vecm)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actual.index, actual.values, color="#1f4e79", lw=2, label="Actual")
ax.plot(join_rdl.index,  join_rdl.values,  color="#c0392b", lw=2, ls="--",
        marker="o", label="ARDL/NARDL ensemble (OBR-conditioned)")
ax.plot(join_vecm.index, join_vecm.values, color="#e08e0b", lw=2, ls=":",
        marker="s", label="VECM (unconditional system)")
ax.set_ylabel("Net additional dwellings")
ax.set_xlabel("Financial year (start)")
ax.legend()
plt.tight_layout()
plt.show()

# Chronos

In [ ]:

p, seasonal, fy_map = B["p"], B["seasonal"], B["fy_map"]
net_add, actual_back = B["net_add"], B["actual_back"]

fc_chr = pd.read_csv("../../data/outputs/forecasts/chronos_forward.csv")
fc_chr["log_starts"] = np.log(fc_chr["starts"])

ln_S_chr = np.concatenate([B["seed"], fc_chr["log_starts"].values])

quarters_chr = pd.PeriodIndex(fc_chr["Quarter"], freq="Q")

ln_C_chr, lag_C = [], p["last_ln_C"]   # reset seed -- don't reuse mutated lag_C
for t in range(len(fc_chr)):
    lag_C = p["intercept"] + p["rho"]*lag_C + p["beta"]*ln_S_chr[t] + seasonal[quarters_chr[t].quarter]
    ln_C_chr.append(lag_C)
fc_chr["completions"] = np.exp(ln_C_chr) * p["smearing_factor"]
fc_chr["fy"] = quarters_chr.map(fy_map)

annual_chr = fc_chr.groupby("fy")["completions"].sum().rename("private_completions").to_frame().iloc[1:]

annual_chr["net_additions"] = net_add(annual_chr["private_completions"])

priv_2025_26_chr = actual_back + fc_chr.loc[fc_chr["Quarter"] == "2026Q1", "completions"].iloc[0]

delivery_chr = {"2024-25": 208600, "2025-26": net_add(priv_2025_26_chr),
                 "2026-27": annual_chr.loc[2026, "net_additions"],
                 "2027-28": annual_chr.loc[2027, "net_additions"],
                 "2028-29": annual_chr.loc[2028, "net_additions"]}

cumulative_chr = sum(delivery_chr.values())
print("--- Chronos (unconditional) ---")
for fy, v in delivery_chr.items():
    print(f"{fy}: {v:,.0f}")
print(f"\nCumulative: {cumulative_chr:,.0f} ({100*cumulative_chr/target:.1f}% of {target:,})")

In [ ]:
import matplotlib.pyplot as plt
fc_chr.plot(x="Quarter", y="starts", marker="o", figsize=(8,4))
plt.title("Chronos forward starts, 2026Q1-2029Q1")
plt.show()
print(fc_chr[["Quarter","starts"]])